# 01 - Cleaning the raw data

**Goal:** turn every raw download into one tidy table with the same columns (`date`, `region`), correct units
and no duplicates. Missing values stay empty - we never invent numbers.

Every section follows the same pattern: **look at the raw file -> clean it -> check the result.**
The cleaning functions live in `Src/processing/clean_data.py`; this notebook calls them step by step so you can
see and explain what each one does. The pipeline script runs the same functions without the notebook.

| Output table | Used for |
|---|---|
| `static_context` | soil, terrain, cropland - one row per region |
| `rainfall_daily` | CHIRPS rain - **labels and features** |
| `weather_daily` | NASA POWER + VPD + ET0 - **features** |
| `soil_moisture_daily` | ERA5-Land (features), SMAP (validation) |
| `era5_weather_daily`, `imerg_daily`, `vegetation` | **validation only** |

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Find ML/Src/processing from wherever the notebook is opened (ML/Notebooks or the project root)
here = Path.cwd().resolve()
candidates = [here, here.parent, here / "ML", here.parent / "ML", *here.parents]
PROC_SRC = next((c / "Src" / "processing" for c in candidates
                 if (c / "Src" / "processing" / "pipeline_common.py").exists()), None)
assert PROC_SRC is not None, "Could not find Src/processing - open this notebook from the ML/Notebooks folder."
sys.path.insert(0, str(PROC_SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pipeline_common import (RAW_DIR, PROC_DIR, DATA_DIR, REGIONS, REGION_NAMES,
                             read_region_files, read_static_file, load_table, save_table, longest_run)
import label_config as cfg

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
COLORS = {"Arsi": "#1f77b4", "Bale": "#d62728", "Jimma": "#2ca02c"}
FIG_DIR = DATA_DIR.parent / "Figures"
FIG_DIR.mkdir(exist_ok=True)

def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")

print("Raw data :", RAW_DIR)
print("Processed:", PROC_DIR)
print("Figures  :", FIG_DIR)

In [ ]:
import clean_data as cd

## 1. What did we download?
A quick inventory of every raw file: rows, columns and date range. Anything missing or with a strange date range shows up here.

In [ ]:
rows = []
for folder in sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir()):
    for f in sorted((RAW_DIR / folder).glob("*.csv")):
        df = pd.read_csv(f)
        row = {"dataset": folder, "file": f.name, "rows": len(df), "columns": df.shape[1]}
        if "date" in df.columns:
            d = pd.to_datetime(df["date"])
            row["first_date"], row["last_date"] = d.min().date(), d.max().date()
        rows.append(row)
inventory = pd.DataFrame(rows)
inventory

## 2. Static context (soil, terrain, cropland)
These do not change from day to day, so each region gets **one row**.

* **SRTM** gives elevation and slope.
* **SoilGrids** gives clay, sand, silt, organic carbon and bulk density at three depths. Earth Engine stores them in
  "mapped units", so we convert: clay/sand/silt g/kg -> %, soil carbon dg/kg -> g/kg, bulk density cg/cm3 -> g/cm3.
* **WorldCover** gives the fraction of the 25 km area that is cropland.

In [ ]:
for folder, prefix in [("SRTM", "srtm"), ("SOILGRIDS", "soilgrids"), ("WORLDCOVER", "worldcover")]:
    print(folder)
    display(read_static_file(folder, prefix).set_index("region").T.head(8))

In [ ]:
static = cd.clean_static()

**Check yourself:** elevation should be roughly 1,700-3,000 m, clay between 0 and 100 %, cropland fraction between 0 and 1.
Only 3 regions means these values take only 3 different numbers - they behave like a region label, not like a general soil effect.

## 3. Rainfall (CHIRPS)
CHIRPS is the backbone of the project: it defines the onset trigger, the dry spell and the label.

In [ ]:
raw_chirps = cd.load_chirps()
display(raw_chirps.head())
print("rows:", len(raw_chirps))
print("negative values:", int((raw_chirps["precipitation"] < 0).sum()))
print("duplicate region-days:", int(raw_chirps.duplicated(["region", "date"]).sum()))
print("empty values:", int(raw_chirps["precipitation"].isna().sum()))

In [ ]:
rain = cd.clean_rainfall()
rain.head()

The cleaned table has **one row per region per calendar day** (`complete_grid`). A day that was missing would appear as an empty row,
so later steps never silently skip a day. `wet_day` is 1 when rain >= the wet-day threshold in `label_config.py`.

In [ ]:
days = rain.assign(year=rain["date"].dt.year).groupby(["region", "year"])["rain_mm"].agg(["sum", "count"])
annual = days[days["count"] >= 365]["sum"].unstack(0)

ax = annual.plot(figsize=(10, 3.5), color=[COLORS[c] for c in annual.columns])
ax.set_title("Annual rainfall (CHIRPS), complete years only")
ax.set_ylabel("mm per year")
plt.show()

## 4. Weather (NASA POWER) and evaporative demand
POWER provides temperature, humidity, wind and solar radiation. From them we calculate two numbers that describe **how fast water leaves the soil**:

* **VPD** (vapor pressure deficit) = saturation vapor pressure - actual vapor pressure. High VPD = thirsty air.
* **ET0** (reference evapotranspiration, FAO-56 Penman-Monteith) = how many mm of water the atmosphere can take per day.

Two details handled in the code: radiation units differ between POWER communities (MJ or kWh per m2), so the code detects the unit,
and wind is converted from 10 m to 2 m height for ET0.

In [ ]:
raw_power = read_region_files("NASA_POWER", "nasa_power")
display(raw_power.head())
display(raw_power.describe().loc[["count", "min", "mean", "max"]].round(2))

In [ ]:
weather = cd.clean_weather(static)
weather.head()

In [ ]:
sample = weather.dropna(subset=["tmax_c", "et0_mm"]).sample(4000, random_state=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for region, g in sample.groupby("region"):
    axes[0].scatter(g["tmax_c"], g["vpd_kpa"], s=4, alpha=0.4, color=COLORS[region], label=region)
    axes[1].scatter(g["rad_mj"], g["et0_mm"], s=4, alpha=0.4, color=COLORS[region], label=region)
axes[0].set(xlabel="Tmax (C)", ylabel="VPD (kPa)", title="Hotter days -> higher VPD")
axes[1].set(xlabel="solar radiation (MJ/m2/day)", ylabel="ET0 (mm/day)", title="More sun -> higher ET0")
axes[0].legend(markerscale=3)
plt.show()

**Check yourself:** VPD usually 0.2-2 kPa and ET0 2-6 mm/day in the highlands. Values far outside that mean a unit problem upstream.

## 5. Soil moisture (ERA5-Land and SMAP)
* **ERA5-Land** soil water layers cover the **whole history (1981 onward)**, so they are the model input. Layer 1 is 0-7 cm, layer 2 is 7-28 cm,
  layer 3 is 28-100 cm.
* **SMAP L3** is a satellite observation of the top ~5 cm (2015 onward, with gaps on days the satellite does not see the area).
* **SMAP L4** is a model-based surface and root-zone product (2015 onward, no gaps).

SMAP is kept in separate columns and used to **validate** ERA5-Land, not mixed into it.

In [ ]:
soil = cd.clean_soil_moisture()
soil.dropna(subset=["smap_l4_surface"]).head()

In [ ]:
# How often is SMAP L3 empty, per year? (a steady rate = normal satellite gaps; a year at 1.0 = a download problem)
smap_era = soil[soil["date"] >= "2015-03-31"]
empty_by_year = (smap_era.assign(year=smap_era["date"].dt.year)
                 .groupby("year")[["smap_l3_am", "smap_l3_pm", "smap_l4_surface"]]
                 .agg(lambda s: s.isna().mean()).round(2))
empty_by_year

## 6. Validation-only tables
These are cleaned now but are **not** model inputs. They are used later to check the model's answers.

* `era5_weather_daily` - ERA5-Land weather in friendly units (Kelvin -> C, metres -> mm, wind U/V -> speed)
* `imerg_daily` - a second rainfall source
* `vegetation` - MODIS and Sentinel-2 NDVI/EVI/NDRE/NDMI. Sentinel-2 has several tiles per date, so they are averaged by date.

In [ ]:
cd.clean_era5_weather()
cd.clean_imerg()
cd.clean_vegetation()

## 7. Final check

In [ ]:
summary = []
for f in sorted(PROC_DIR.glob("*.csv")):
    d = pd.read_csv(f)
    summary.append({"table": f.stem, "rows": len(d), "columns": d.shape[1],
                    "empty_cells_%": round(d.isna().mean().mean() * 100, 1)})
pd.DataFrame(summary)

**Next:** `02_merging.ipynb` joins these tables into the daily, seasonal and yearly datasets.